# Transformada de Fourier de una línea vertical

**Procesamiento de Imágenes — Universidad Diego Portales**

Se construye una imagen binaria (fondo negro, línea vertical blanca en el centro)
y se calcula su transformada de Fourier 2D. El espectro se visualiza en 2D
(como imagen) y en 3D (como superficie).

## 1. Preparación y parámetros

Todos los parámetros están en esta celda: tamaño de la imagen y grosor de la línea.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registra la proyección 3D)

# ---- Parámetros ----
ALTO = 500          # filas (M)
ANCHO = 500         # columnas (N)
GROSOR = 1          # grosor de la línea en píxeles
ZOOM = 25           # semiancho (en muestras) de la vista ampliada del espectro

plt.rcParams["image.cmap"] = "gray"

## 2. Imagen binaria

La línea ocupa `GROSOR` columnas centradas en `ANCHO // 2`, a lo alto de toda la imagen.

In [ ]:
def imagen_linea_vertical(alto, ancho, grosor=1):
    # Imagen binaria (0 = negro, 1 = blanco) con una línea vertical centrada
    img = np.zeros((alto, ancho), dtype=np.float64)
    c0 = ancho // 2 - grosor // 2
    img[:, c0:c0 + grosor] = 1.0
    return img

img = imagen_linea_vertical(ALTO, ANCHO, GROSOR)

plt.figure(figsize=(5, 5))
plt.imshow(img, vmin=0, vmax=1)
plt.title(f"Imagen {ALTO}x{ANCHO}, línea de {GROSOR} px")
plt.axis("off")
plt.show()

## 3. Transformada de Fourier

- `np.fft.fft2` calcula la DFT 2D.
- `np.fft.fftshift` mueve la frecuencia cero (componente DC) al centro.
- Se visualiza el **espectro de magnitud** con escala logarítmica, $\log(1 + |F(u,v)|)$,
  porque el término DC domina y ocultaría todo lo demás.

In [ ]:
F = np.fft.fftshift(np.fft.fft2(img))
magnitud = np.abs(F)
espectro = np.log1p(magnitud)

# Ejes de frecuencia (ciclos/píxel), centrados en 0
u = np.fft.fftshift(np.fft.fftfreq(ANCHO))   # frecuencia horizontal
v = np.fft.fftshift(np.fft.fftfreq(ALTO))    # frecuencia vertical

print("Componente DC |F(0,0)| =", magnitud[ALTO // 2, ANCHO // 2])
print("Suma de píxeles blancos =", img.sum())

## 4. Visualización 2D

En el espectro completo, una fila de 1 px en 500 px puede perderse al escalar la figura a la
pantalla; por eso el tercer panel muestra una zona ampliada alrededor del origen.

In [ ]:
ext = [u[0], u[-1], v[-1], v[0]]
cf, cc = ALTO // 2, ANCHO // 2
zona = (slice(cf - ZOOM, cf + ZOOM + 1), slice(cc - ZOOM, cc + ZOOM + 1))
ext_zoom = [u[cc - ZOOM], u[cc + ZOOM], v[cf + ZOOM], v[cf - ZOOM]]

fig, ax = plt.subplots(1, 3, figsize=(16, 5))

ax[0].imshow(img, vmin=0, vmax=1, interpolation="nearest")
ax[0].set_title("Imagen original")
ax[0].axis("off")

im1 = ax[1].imshow(espectro, extent=ext, cmap="gray", interpolation="nearest")
ax[1].set_title(r"Espectro $\log(1+|F(u,v)|)$")
ax[1].set_xlabel("u (ciclos/px)")
ax[1].set_ylabel("v (ciclos/px)")
fig.colorbar(im1, ax=ax[1], fraction=0.046)

im2 = ax[2].imshow(espectro[zona], extent=ext_zoom, cmap="inferno", interpolation="nearest")
ax[2].set_title(f"Espectro ampliado (±{ZOOM} muestras del centro)")
ax[2].set_xlabel("u (ciclos/px)")
ax[2].set_ylabel("v (ciclos/px)")
fig.colorbar(im2, ax=ax[2], fraction=0.046)

plt.tight_layout()
plt.show()

Perfil del espectro a lo largo de la fila central ($v = 0$) y de la columna central ($u = 0$):

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(u, magnitud[ALTO // 2, :])
ax[0].set_title("|F(u, 0)|  (fila central)")
ax[0].set_xlabel("u (ciclos/px)")
ax[0].grid(alpha=0.3)

ax[1].plot(v, magnitud[:, ANCHO // 2])
ax[1].set_title("|F(0, v)|  (columna central)")
ax[1].set_xlabel("v (ciclos/px)")
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Visualización 3D

Se grafica $\log(1+|F(u,v)|)$ como superficie. Con 500x500 puntos el render es lento,
así que `PASO` submuestrea la malla solo para el dibujo.
Para rotar el gráfico con el mouse, ejecuta antes `%matplotlib widget` (requiere `ipympl`)
o `%matplotlib qt`.

In [ ]:
PASO = 2   # submuestreo para el gráfico 3D (1 = todos los puntos)

def indices_con_centro(n, paso):
    # Índices cada `paso` muestras que siempre incluyen el centro n // 2
    # (si no, el submuestreo podría saltarse justo la fila de la línea)
    c = n // 2
    return np.r_[np.arange(c, -1, -paso)[::-1], np.arange(c + paso, n, paso)]

filas = indices_con_centro(ALTO, PASO)
cols = indices_con_centro(ANCHO, PASO)
U, V = np.meshgrid(u[cols], v[filas])
malla = np.ix_(filas, cols)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(U, V, espectro[malla],
                cmap="inferno", linewidth=0, antialiased=False)
ax.set_title(r"Espectro 3D: $\log(1+|F(u,v)|)$")
ax.set_xlabel("u (ciclos/px)")
ax.set_ylabel("v (ciclos/px)")
ax.set_zlabel("log(1+|F|)")
ax.view_init(elev=30, azim=-60)
plt.tight_layout()
plt.show()

La misma superficie en magnitud lineal, $|F(u,v)|$:

In [ ]:
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(U, V, magnitud[malla],
                cmap="viridis", linewidth=0, antialiased=False)
ax.set_title("Espectro 3D: |F(u,v)| (escala lineal)")
ax.set_xlabel("u (ciclos/px)")
ax.set_ylabel("v (ciclos/px)")
ax.set_zlabel("|F|")
ax.view_init(elev=30, azim=-60)
plt.tight_layout()
plt.show()

## 6. Interpretación

La imagen no cambia en dirección vertical: cada columna es constante. Entonces:

$$f(x,y) = g(x)\cdot 1 \quad\Longrightarrow\quad F(u,v) = G(u)\,\cdot\,M\,\delta(v)$$

- En la dirección **vertical** ($v$) solo existe la frecuencia cero, así que toda la energía cae en la fila $v = 0$.
- En la dirección **horizontal** ($u$), $g(x)$ es un pulso angosto. Un impulso de 1 px tiene espectro
  de **magnitud constante**: $|F(u,0)| = M$ para todo $u$.

**Una línea vertical en la imagen produce una línea horizontal en el espectro**, perpendicular a la original.

Al aumentar `GROSOR`, el pulso se ensancha y su transformada pasa a ser una función tipo sinc
(núcleo de Dirichlet) cada vez más angosta. La siguiente celda lo muestra.

In [ ]:
grosores = [1, 5, 20]
fig, ax = plt.subplots(2, len(grosores), figsize=(15, 8))

for j, g in enumerate(grosores):
    im_g = imagen_linea_vertical(ALTO, ANCHO, g)
    mag_g = np.abs(np.fft.fftshift(np.fft.fft2(im_g)))

    ax[0, j].imshow(np.log1p(mag_g)[zona], extent=ext_zoom, cmap="inferno",
                    interpolation="nearest")
    ax[0, j].set_title(f"Espectro ampliado, grosor = {g} px")
    ax[0, j].set_xlabel("u")
    ax[0, j].set_ylabel("v")

    ax[1, j].plot(u, mag_g[ALTO // 2, :])
    ax[1, j].set_title(f"|F(u, 0)|, grosor = {g} px")
    ax[1, j].set_xlabel("u (ciclos/px)")
    ax[1, j].grid(alpha=0.3)

plt.tight_layout()
plt.show()